<a href="https://colab.research.google.com/github/martine-augustin/Marionnaud/blob/main/Copie_de_marionnaud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Marionnaud projet


## Variable et installation

In [1]:
# Install
!pip install beautifulsoup4
!pip install lxml




In [2]:
# Import
from bs4 import BeautifulSoup
import requests
import pandas as pd
import numpy as np
import time
import random
import csv
import pandas as pd

In [3]:
#Constante
id_avis = 0
page_avis = 0
page_avis_max = 365
page_avis_end = False
liste_des_avis = list()

## Fonction

In [4]:
def unAvis(section):
    global id_avis

    img_star = section.find('img', class_='CDS_StarRating_starRating__614d2e')
    if not img_star:
        return None

    nb_etoile_text = img_star.get('alt')
    nb_etoile = nb_etoile_text[5]

    date_commentaire_tag = section.find('span')
    if not date_commentaire_tag or not nb_etoile.isdigit():
        return None

    date_commentaire = date_commentaire_tag.get_text(strip=True)
    if not date_commentaire[0].isdigit():
        return None

    id_avis += 1
    titre_tag = section.find('h2')
    titre = titre_tag.get_text(strip=True) if titre_tag else None

    commentaire_tag = section.find('p')
    commentaire_text = commentaire_tag.get_text(strip=True) if commentaire_tag else None

    return [id_avis, page_avis, nb_etoile, date_commentaire, titre, commentaire_text]

In [5]:
def PageAvis(url):
    liste_des_avis_page = []

    # Parser le HTML
    soup = BeautifulSoup(response.text, 'html.parser')
    sections = soup.find_all('section')  # toutes les sections

    for section in sections:
        avis = unAvis(section)
        if avis:  # si unAvis renvoie bien une liste
            liste_des_avis_page.append(avis)

    return liste_des_avis_page

In [6]:
def CreateCSV(tab_avis):
  with open("avis.csv", mode="w", newline="", encoding="utf-8") as fichier_csv:
    writer = csv.writer(fichier_csv)

    # Écrire une ligne d'en-tête
    writer.writerow(["Avis ID", "Page ID", "Nombre etoile", "Date", "Titre", "Commentaire"])

    # Écrire les données
    for ligne in tab_avis:
        writer.writerow(ligne)

In [7]:
def OpenCSV(filename):
    """
    Ouvre un fichier CSV et retourne un DataFrame pandas
    """
    try:
        df = pd.read_csv(filename, encoding="utf-8")
        return df
    except FileNotFoundError:
        print(f"Erreur : le fichier {filename} n'existe pas.")
        return None
    except pd.errors.EmptyDataError:
        print(f"Erreur : le fichier {filename} est vide.")
        return None

## Scrapping

In [ ]:
import time
import pandas as pd
import chromedriver_autoinstaller
from selenium import webdriver
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup

#Cette ligne magique télécharge automatiquement le bon driver pour la version de Chrome installée
chromedriver_autoinstaller.install()

#Configuration des options
options = webdriver.ChromeOptions()
options.add_argument('--headless') # Indispensable
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--window-size=1920,1080') # Pour bien voir le tableau

print("🚀 Démarrage du navigateur...")
driver = webdriver.Chrome(options=options)
print("✅ Navigateur lancé avec succès !")
#URL cible
url = "https://fr.trustpilot.com/review/www.marionnaud.com"
print(f"🔗 Connexion à {url}...")
driver.get(url)
time.sleep(5) # Attente chargement initial

all_data = []
page_count = 0
MAX_PAGES = 366 # Sécurité (il y a 366 pages, on met une marge)

try:
    while True:
        page_count += 1
        print(f"📄 Traitement page {page_count}...", end="\r")

#Lecture du tableau
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        # On utilise l'ID précis vu dans tes images
        table = soup.find('table', id='wpgmza_table_1')


#Passage à la page suivante
        try:
            next_button = driver.find_element(By.ID, 'wpgmza_table_1_next')

#Vérification de fin : si le bouton a la classe "disabled", on arrête
            if "disabled" in next_button.get_attribute("class"):
                print(f"\n✅ Fin atteinte à la page {page_count}.")
                break

#Clic et attente
            driver.execute_script("arguments[0].click();", next_button)
            time.sleep(2) # Pause pour laisser le tableau changer

        except Exception as e:
            print(f"\n⚠️ Erreur pagination : {e}")
            break

#Sécurité boucle infinie
        if page_count >= MAX_PAGES:
            print("\n⚠️ Arrêt de sécurité (Max Pages atteint).")
            break

finally:
    driver.quit()
#--- ÉTAPE 3 : SAUVEGARDE ---
print(f"\n🎉 Terminé ! {len(all_data)} profils récupérés.")
df = pd.DataFrame(all_data)


In [30]:
#On recupere les avis de toutes les pages Marionnaud
page_avis= 0
page_avis_end= False

while page_avis_end == False:
  page_avis += 1
  url = 'https://fr.trustpilot.com/review/www.marionnaud.com?page=' + str(page_avis)

  # Récupérer le contenu de la page avec requests
  response = requests.get(url)

  if response.status_code != 200:
    print(f"Error status code : {response.status_code}")
    page_avis_end = True
  else:
    result = PageAvis(url)
    if len(result)> 0:
      if liste_des_avis:
        liste_des_avis = liste_des_avis + result
      else:
        liste_des_avis = result
    else:
      page_avis_end= True

  # Génère un nombre entier aléatoire entre 2 et 5
  delai = random.randint(2, 5)
  print("Début du timeur de la page : " + str(page_avis) + " en cours ...")
  time.sleep(delai)  # attend 5 secondes

  print("--------------------------")
  print("url :" + url)
  print(result)
  print("nombre élément liste_des_avis : " + str(len(liste_des_avis)))

print(liste_des_avis)
CreateCSV(liste_des_avis)

test
Début du timeur de la page : 1 en cours ...
--------------------------
url :https://fr.trustpilot.com/review/www.marionnaud.com?page=1
[[601, 1, '5', '27 février 2026', 'BONJOUR,', "BONJOUR,JE SUIS CONTENTE DE MON PASSAGE A LA BOUTIQUE MARIONNAUD RUE ORDENER , PERSONNELS COMPETENTS , SEVIABLES , AIMABLES , ET AUSSI D'UNE GENTILLESSE SANS PAREIL.JE RECOMMANDE CETTE BOUTIQUE.FORNAS MICHELE"], [602, 1, '5', '24 février 2026', 'Un parfum pour mon petit fils.', "Un parfum pour mon petit fils.C'était une première pour lui, j'avais besoin de conseils et j'ai trouvé une équipe formidable au magasin d'Antony. Ce magasin m'a fait changé d'idée sur Marionnaud (en effet, tous les magasins ne sont pas aussi aimables).BRAVO à cette équipe"], [603, 1, '1', '28 février 2026', 'J’ai passé une commande sur leur site…', 'J’ai passé une commande sur leur site 21/02/2026 , expédiée par Marionnaud. Trois dates de livraison m’ont été données via le transporteur Colissimo, mais le colis n’est jamais arri

In [31]:
# Ouvrir le CSV créé précédemment
df_avis = OpenCSV("avis.csv")

# Vérifier que ça a fonctionné
if df_avis is not None:
    print(df_avis.head())  # affiche les 5 premières lignes


   Avis ID  Page ID  Nombre etoile             Date  \
0        1        1              5  27 février 2026   
1        2        1              5  24 février 2026   
2        3        1              1  28 février 2026   
3        4        1              5  23 février 2026   
4        5        1              5  25 février 2026   

                                    Titre  \
0                                BONJOUR,   
1          Un parfum pour mon petit fils.   
2  J’ai passé une commande sur leur site…   
3                                Le choix   
4                       Une équipe au top   

                                         Commentaire  
0  BONJOUR,JE SUIS CONTENTE DE MON PASSAGE A LA B...  
1  Un parfum pour mon petit fils.C'était une prem...  
2  J’ai passé une commande sur leur site 21/02/20...  
3  Le choix, l accueil et les conseils sont les a...  
4  Une équipe au top! De très bons conseils et un...  
